# FreshMart stockout prototype - Stage 3: stockout risk model + GenAI rationale

Input: `serverless_stable_zlgeaw_catalog.stockout_core.gold_store_sku_daily` (+ `dim_product`, `dim_supplier`, `gold_supplier_performance`). PII columns in `dim_store` are not used.

1. Point-in-time features (past data only) and label = stockout within the next max(3, ceil(supplier lead time)) days, for pairs that are in stock today.
2. Time split: train May-Sep (with a purge gap so labels do not look into October), test October.
3. LightGBM classifier, logged to MLflow and registered as `stockout_ml.stockout_risk_model@champion`; compared with the legacy min/max rule `on_hand <= avg_demand x (lead_time_mean + 3)`.
4. Score the latest snapshot for all 6,000 store-SKU pairs -> `stockout_ml.stockout_predictions`.
5. `ai_query` rationale for the top ~200 at-risk pairs -> `stockout_ml.replenishment_recommendations`.

In [0]:
%pip install -q lightgbm
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import math
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, Window as W

CAT = "serverless_stable_zlgeaw_catalog"
CORE = f"{CAT}.stockout_core"
ML = f"{CAT}.stockout_ml"
MODEL_NAME = f"{ML}.stockout_risk_model"
LLM_ENDPOINT = "system.ai.meta-llama-3-3-70b-instruct"

TEST_START = "2026-10-01"   # last month = test
WARMUP_START = "2026-05-29" # need 28 days of history for rolling features
MAX_HORIZON = 7             # max supplier lead time mean is 7 days

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ML}")

DataFrame[]

In [0]:
gold = spark.table(f"{CORE}.gold_store_sku_daily")
sup = spark.table(f"{CORE}.dim_supplier").select(
    "supplier_id", "lead_time_days_mean", "lead_time_days_sd", F.col("fill_rate").alias("supplier_fill_rate"))
cat_map = (spark.table(f"{CORE}.dim_product").select("category").distinct()
           .withColumn("category_idx", F.dense_rank().over(W.orderBy("category")) - 1))

w = W.partitionBy("store_id", "sku_id").orderBy("date")
def trailing(n):  # last n days incl. today (end-of-day snapshot = known at scoring time)
    return w.rowsBetween(-(n - 1), 0)

so = F.col("is_stockout").cast("int")
instock = 1 - so

feat = (gold
    .join(F.broadcast(sup), "supplier_id", "left")
    .join(F.broadcast(cat_map), "category", "left")
    .withColumn("so_int", so)
    .withColumn("avg_units_7d", F.avg("units_sold").over(trailing(7)))
    .withColumn("avg_units_28d", F.avg("units_sold").over(trailing(28)))
    .withColumn("std_units_28d", F.coalesce(F.stddev("units_sold").over(trailing(28)), F.lit(0.0)))
    # demand on in-stock days only (sales are censored on stockout days)
    .withColumn("avg_demand_28d", F.sum("units_sold").over(trailing(28)) /
                F.greatest(F.sum(instock).over(trailing(28)), F.lit(1)))
    .withColumn("stockout_days_28d", F.sum("so_int").over(trailing(28)))
    .withColumn("last_so_date", F.max(F.when(so == 1, F.col("date"))).over(w.rowsBetween(W.unboundedPreceding, -1)))
    .withColumn("days_since_stockout", F.coalesce(F.datediff("date", "last_so_date"), F.lit(999)))
    .withColumn("on_hand_change_1d", F.col("on_hand_units") - F.lag("on_hand_units", 1).over(w))
    .withColumn("on_hand_change_7d", F.col("on_hand_units") - F.lag("on_hand_units", 7).over(w))
    .withColumn("_d", F.greatest(F.col("avg_demand_28d"), F.lit(0.1)))
    .withColumn("days_of_cover", F.col("on_hand_units") / F.col("_d"))
    .withColumn("days_of_cover_incl_on_order", (F.col("on_hand_units") + F.col("on_order_units")) / F.col("_d"))
    .withColumn("cover_minus_lead_time", F.col("days_of_cover") - F.col("lead_time_days_mean"))
    .withColumn("promo_adj_cover", F.col("on_hand_units") / (F.col("_d") * F.when(F.col("on_promo"), F.col("promo_lift")).otherwise(1.0)))
    .withColumn("velocity_trend", F.col("avg_units_7d") / F.greatest(F.col("avg_units_28d"), F.lit(0.1)))
    .withColumn("fill_to_capacity", F.col("on_hand_units") / F.col("shelf_capacity"))
    .withColumn("on_promo_int", F.col("on_promo").cast("int"))
    .withColumn("is_perishable_int", F.col("is_perishable").cast("int"))
    .withColumn("day_of_week", F.dayofweek("date"))
    # legacy min/max rule
    .withColumn("legacy_reorder_point", F.col("avg_units_28d") * (F.col("lead_time_days_mean") + 3))
    .withColumn("legacy_flag", (F.col("on_hand_units") <= F.col("legacy_reorder_point")).cast("int"))
    .withColumn("legacy_score", F.col("legacy_reorder_point") / (F.col("on_hand_units") + 1))
    # label: stockout in the next H = max(3, ceil(lead time)) days
    .withColumn("horizon_days", F.greatest(F.lit(3), F.ceil("lead_time_days_mean")).cast("int"))
    .withColumn("label", F.greatest(*[
        F.when(F.lit(k) <= F.col("horizon_days"), F.coalesce(F.lead("so_int", k).over(w), F.lit(0))).otherwise(0)
        for k in range(1, MAX_HORIZON + 1)]))
    .withColumn("label_end_date", F.date_add("date", F.col("horizon_days")))
    .drop("_d", "last_so_date")
)

feat.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{ML}.stockout_features")
feat = spark.table(f"{ML}.stockout_features")
display(feat.agg(F.count("*").alias("rows"), F.min("date"), F.max("date"), F.avg("so_int").alias("stockout_rate"),
                 F.avg("label").alias("label_rate_all")))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rows,min(date),max(date),stockout_rate,label_rate_all
1104000,2026-05-01,2026-10-31,0.07083876811594203,0.17126539855072465


In [0]:
FEATURES = [
    "on_hand_units", "on_order_units", "units_sold", "avg_units_7d", "avg_units_28d", "std_units_28d", "avg_demand_28d",
    "days_of_cover", "days_of_cover_incl_on_order", "cover_minus_lead_time", "promo_adj_cover", "velocity_trend",
    "fill_to_capacity", "on_hand_change_1d", "on_hand_change_7d", "stockout_days_28d", "days_since_stockout",
    "lead_time_days_mean", "lead_time_days_sd", "supplier_fill_rate", "on_promo_int", "promo_lift",
    "day_of_week", "is_perishable_int", "category_idx", "case_pack", "shelf_capacity", "unit_price",
]
max_date = feat.agg(F.max("date")).first()[0]

# in-stock today (predict NEW stockouts), enough history, label window fully observed
base = feat.where((F.col("so_int") == 0) & (F.col("date") >= WARMUP_START) & (F.col("label_end_date") <= F.lit(max_date)))
# purge: training label windows must end before the test month starts
train_sdf = base.where(F.col("label_end_date") < F.lit(TEST_START))
test_sdf = base.where(F.col("date") >= F.lit(TEST_START))

cols = ["date", "store_id", "sku_id", "label", "legacy_flag", "legacy_score"] + FEATURES
train_pd = train_sdf.select(*cols).toPandas()
test_pd = test_sdf.select(*cols).toPandas()
for d in (train_pd, test_pd):
    d[FEATURES] = d[FEATURES].astype("float64")
print(f"train: {len(train_pd):,} rows {train_pd.date.min()}..{train_pd.date.max()} pos_rate={train_pd.label.mean():.4f}")
print(f"test:  {len(test_pd):,} rows {test_pd.date.min()}..{test_pd.date.max()} pos_rate={test_pd.label.mean():.4f}")
assert train_pd.date.max() < test_pd.date.min()

train: 672,452 rows 2026-05-29..2026-09-27 pos_rate=0.1296
test:  146,802 rows 2026-10-01..2026-10-28 pos_rate=0.1432


In [0]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

params = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=200,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1)
model = lgb.LGBMClassifier(**params)
model.fit(train_pd[FEATURES], train_pd["label"])

y = test_pd["label"].values
p_model = model.predict_proba(test_pd[FEATURES])[:, 1]

def precision_at_top(y, s, frac=0.05):
    k = int(math.ceil(len(s) * frac))
    idx = np.argsort(-s, kind="stable")[:k]
    return float(y[idx].mean())

metrics = {
    "test_auc": roc_auc_score(y, p_model),
    "test_pr_auc": average_precision_score(y, p_model),
    "test_precision_at_top5pct": precision_at_top(y, p_model),
    "legacy_auc": roc_auc_score(y, test_pd["legacy_score"]),
    "legacy_flag_auc": roc_auc_score(y, test_pd["legacy_flag"]),
    "legacy_pr_auc": average_precision_score(y, test_pd["legacy_score"]),
    "legacy_precision_at_top5pct": precision_at_top(y, test_pd["legacy_score"].values),
    "legacy_flag_precision": precision_score(y, test_pd["legacy_flag"]),
    "legacy_flag_recall": recall_score(y, test_pd["legacy_flag"]),
    "legacy_flag_rate": float(test_pd["legacy_flag"].mean()),
    "test_positive_rate": float(y.mean()),
    "train_rows": float(len(train_pd)), "test_rows": float(len(test_pd)),
}
# model precision/recall when flagging the same share of rows as the legacy rule
k = int(round(metrics["legacy_flag_rate"] * len(y)))
thr = np.sort(p_model)[::-1][k - 1]
flag_m = (p_model >= thr).astype(int)
metrics["model_precision_at_legacy_rate"] = precision_score(y, flag_m)
metrics["model_recall_at_legacy_rate"] = recall_score(y, flag_m)

display(pd.DataFrame([(k_, round(v, 4)) for k_, v in metrics.items()], columns=["metric", "value"]))
imp = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
display((imp / imp.sum()).round(4).rename("gain_share").reset_index().rename(columns={"index": "feature"}).head(15))
print("test score quantiles:", np.round(np.quantile(p_model, [0, .25, .5, .75, .9, .95, .99, 1]), 4))

metric,value
test_auc,0.9213
test_pr_auc,0.6814
test_precision_at_top5pct,0.8218
legacy_auc,0.8482
legacy_flag_auc,0.7439
legacy_pr_auc,0.4836
legacy_precision_at_top5pct,0.6338
legacy_flag_precision,0.2607
legacy_flag_recall,0.9268
legacy_flag_rate,0.5089


feature,gain_share
cover_minus_lead_time,0.5602
on_order_units,0.0808
lead_time_days_mean,0.0683
promo_adj_cover,0.0521
promo_lift,0.0319
on_promo_int,0.017
velocity_trend,0.0165
on_hand_units,0.0161
lead_time_days_sd,0.0151
avg_units_28d,0.015


test score quantiles: [0.     0.0021 0.0219 0.17   0.4678 0.6683 0.8896 0.9884]


In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/retail-stockout-prototype/stockout_risk_experiment")

X_ex = test_pd[FEATURES].head(200)
signature = infer_signature(X_ex, model.predict_proba(X_ex)[:, 1])
with mlflow.start_run(run_name="lgbm_stockout_risk") as run:
    mlflow.log_params({**params, "features": ",".join(FEATURES), "label": "stockout within max(3, ceil(lead_time_mean)) days",
                       "train_end_label_date": "< " + TEST_START, "test_start": TEST_START})
    mlflow.log_metrics(metrics)
    mlflow.log_table((imp / imp.sum()).rename("gain_share").reset_index().rename(columns={"index": "feature"}), "feature_importance.json")
    mlflow.sklearn.log_model(model, artifact_path="model", signature=signature, input_example=X_ex.head(5),
                             registered_model_name=MODEL_NAME, pyfunc_predict_fn="predict_proba")
    run_id = run.info.run_id

client = MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'") if v.run_id == run_id)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.update_registered_model(MODEL_NAME, description="LightGBM stockout risk: P(stockout within max(3, ceil(supplier lead time)) days) for in-stock store-SKU pairs. Trained May-Sep 2026, tested Oct 2026.")
print(f"Registered {MODEL_NAME} v{version} @champion (run {run_id})")

2026/11/02 22:48:12 INFO mlflow.tracking.fluent: Experiment with name '/Users/sabir.ayappalli@databricks.com/retail-stockout-prototype/stockout_risk_experiment' does not exist. Creating a new experiment.
2026/11/02 22:48:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://fevm-serverless-stable-zlgeaw.cloud.databricks.com/ml/experiments/173068929103000/models/m-47c495491a314455ac9de1d7bd1fa65a?o=7474653766284409
Successfully registered model 'serverless_stable_zlgeaw_catalog.stockout_ml.stockout_risk_model'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'serverless_stable_zlgeaw_catalog.stockout_ml.stockout_risk_model': https://fevm-serverless-stable-zlgeaw.cloud.databricks.com/explore/data/models/serverless_stable_zlgeaw_catalog/stockout_ml/stockout_risk_model/version/1?o=7474653766284409


Registered serverless_stable_zlgeaw_catalog.stockout_ml.stockout_risk_model v1 @champion (run fb19aca72e6d4eb19715e3e970004599)


In [0]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
champion_version = client.get_model_version_by_alias(MODEL_NAME, "champion").version

perf = spark.table(f"{CORE}.gold_supplier_performance").select(
    "supplier_id", "avg_actual_lead_time_days", "stddev_lead_time_days", "on_time_rate", "avg_fill_ratio")
snap_cols = ["date", "store_id", "store_name", "region", "sku_id", "sku_name", "category", "supplier_id", "supplier_name",
             "is_stockout", "case_pack", "shelf_capacity", "on_promo", "horizon_days", "legacy_flag"]
snap = (feat.where(F.col("date") == F.lit(max_date)).join(F.broadcast(perf), "supplier_id", "left")
        .select(*dict.fromkeys(snap_cols + FEATURES + ["avg_actual_lead_time_days", "stddev_lead_time_days", "on_time_rate", "avg_fill_ratio"])).toPandas())
X = snap[FEATURES].astype("float64")
snap["model_score"] = champion.predict_proba(X)[:, 1]

# key drivers from LightGBM SHAP contributions (top 3 features pushing risk up)
contrib = champion.predict(X, pred_contrib=True)[:, :len(FEATURES)]
LABELS = {
    "days_of_cover": lambda r: f"only {r.days_of_cover:.1f} days of cover",
    "days_of_cover_incl_on_order": lambda r: f"{r.days_of_cover_incl_on_order:.1f} days of cover incl. on-order",
    "cover_minus_lead_time": lambda r: f"cover {r.cover_minus_lead_time:+.1f} days vs lead time",
    "promo_adj_cover": lambda r: f"{r.promo_adj_cover:.1f} days of promo-adjusted cover",
    "on_hand_units": lambda r: f"low on-hand ({int(r.on_hand_units)} units)",
    "on_order_units": lambda r: f"on-order {int(r.on_order_units)} units",
    "velocity_trend": lambda r: f"sales velocity {r.velocity_trend:.2f}x of 28d avg",
    "avg_units_7d": lambda r: f"7d sales {r.avg_units_7d:.1f}/day",
    "avg_demand_28d": lambda r: f"demand {r.avg_demand_28d:.1f}/day",
    "lead_time_days_mean": lambda r: f"supplier lead time {r.lead_time_days_mean:.1f} days",
    "lead_time_days_sd": lambda r: f"lead time variability (sd {r.lead_time_days_sd:.1f} days)",
    "supplier_fill_rate": lambda r: f"supplier fill rate {r.supplier_fill_rate:.0%}",
    "on_promo_int": lambda r: "active promotion" if r.on_promo_int else "no promotion",
    "promo_lift": lambda r: f"promo lift {r.promo_lift:.2f}x",
    "stockout_days_28d": lambda r: f"{int(r.stockout_days_28d)} stockout days in last 28",
    "days_since_stockout": lambda r: f"last stockout {int(r.days_since_stockout)} days ago" if r.days_since_stockout < 999 else "no recent stockout",
    "is_perishable_int": lambda r: "perishable item" if r.is_perishable_int else "non-perishable item",
}
def driver_text(r, f):
    return LABELS[f](r) if f in LABELS else f"{f}={getattr(r, f):.2f}"

drivers = []
for i, r in enumerate(snap.itertuples(index=False)):
    if r.is_stockout:
        drivers.append(["currently out of stock"] + ([f"on-order {int(r.on_order_units)} units"] if r.on_order_units > 0 else ["nothing on order"]))
        continue
    order = np.argsort(-contrib[i])[:3]
    drivers.append([driver_text(r, FEATURES[j]) for j in order if contrib[i, j] > 0])
snap["key_drivers"] = drivers

# risk score: pairs already out of stock are certain (model is trained on in-stock pairs)
snap["risk_score"] = np.where(snap["is_stockout"], 1.0, snap["model_score"]).round(4)
snap["risk_tier"] = np.select(
    [snap["is_stockout"], snap["risk_score"] >= 0.6, snap["risk_score"] >= 0.3, snap["risk_score"] >= 0.1],
    ["Out of stock", "High", "Medium", "Low"], default="Minimal")

# recommended order: demand over lead time + review day + safety stock from demand & lead-time variability
Z = 1.65  # ~95% cycle service level
d = snap["avg_demand_28d"] * np.where(snap["on_promo"], snap["promo_lift"], 1.0)
L = snap["avg_actual_lead_time_days"].fillna(snap["lead_time_days_mean"])
sL = snap["stddev_lead_time_days"].fillna(snap["lead_time_days_sd"])
snap["safety_stock_units"] = (Z * np.sqrt(L * snap["std_units_28d"] ** 2 + d ** 2 * sL ** 2)).round(1)
snap["target_stock_units"] = (d * (L + 1) + snap["safety_stock_units"]).round(1)
need = (snap["target_stock_units"] - snap["on_hand_units"] - snap["on_order_units"]).clip(lower=0)
raw_order = np.ceil(need / snap["case_pack"]) * snap["case_pack"]
room = (np.floor((snap["shelf_capacity"] - snap["on_hand_units"] - snap["on_order_units"]).clip(lower=0) / snap["case_pack"]) * snap["case_pack"])
snap["recommended_order_units"] = np.minimum(raw_order, room).astype(int)
snap["capped_by_shelf"] = raw_order > room

snap["model_name"] = MODEL_NAME
snap["model_version"] = str(champion_version)
out_cols = ["date", "store_id", "store_name", "region", "sku_id", "sku_name", "category", "supplier_id", "supplier_name",
            "on_hand_units", "on_order_units", "avg_demand_28d", "avg_units_7d", "days_of_cover", "on_promo", "promo_lift",
            "lead_time_days_mean", "lead_time_days_sd", "avg_actual_lead_time_days", "horizon_days", "case_pack", "shelf_capacity",
            "is_stockout", "legacy_flag", "model_score", "risk_score", "risk_tier", "safety_stock_units", "target_stock_units",
            "recommended_order_units", "capped_by_shelf", "key_drivers", "model_name", "model_version"]
pred_sdf = (spark.createDataFrame(snap[out_cols])
            .withColumnRenamed("date", "snapshot_date")
            .withColumn("scored_at", F.current_timestamp()))
pred_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{ML}.stockout_predictions")
spark.sql(f"COMMENT ON TABLE {ML}.stockout_predictions IS 'Stockout risk for every store-SKU pair on the latest snapshot date, scored by {MODEL_NAME}@champion. risk_score = P(stockout within horizon_days) (1.0 if already out of stock); recommended_order_units = demand over lead time + 1 review day + safety stock (z=1.65), rounded up to case_pack and capped at shelf_capacity.'")
print("scored", len(snap), "pairs for", max_date)

scored 6000 pairs for 2026-10-31


In [0]:
%sql
SELECT count(*) AS rows, count(DISTINCT store_id, sku_id) AS unique_pairs, min(snapshot_date) AS snap_min, max(snapshot_date) AS snap_max,
       round(avg(risk_score), 4) AS avg_risk, round(percentile(model_score, 0.5), 4) AS p50_model, round(percentile(model_score, 0.95), 4) AS p95_model,
       sum(CASE WHEN recommended_order_units % case_pack <> 0 THEN 1 ELSE 0 END) AS bad_case_pack,
       sum(CASE WHEN on_hand_units + on_order_units + recommended_order_units > shelf_capacity AND recommended_order_units > 0 THEN 1 ELSE 0 END) AS over_capacity
FROM serverless_stable_zlgeaw_catalog.stockout_ml.stockout_predictions;

rows,unique_pairs,snap_min,snap_max,avg_risk,p50_model,p95_model,bad_case_pack,over_capacity
6000,6000,2026-10-31,2026-10-31,0.2011,0.0309,0.8672,0,0


In [0]:
%sql
SELECT risk_tier, count(*) AS pairs, round(avg(risk_score), 3) AS avg_score, round(avg(days_of_cover), 2) AS avg_cover,
       sum(recommended_order_units) AS total_order_units, sum(legacy_flag) AS legacy_flagged
FROM serverless_stable_zlgeaw_catalog.stockout_ml.stockout_predictions
GROUP BY risk_tier ORDER BY avg_score DESC;

risk_tier,pairs,avg_score,avg_cover,total_order_units,legacy_flagged
Out of stock,481,1.0,0.0,3630,481
High,334,0.747,1.51,1920,333
Medium,588,0.437,2.65,1398,578
Low,844,0.185,4.02,1332,808
Minimal,3753,0.017,7.79,9480,1363


In [0]:
%sql
CREATE OR REPLACE TABLE serverless_stable_zlgeaw_catalog.stockout_ml.replenishment_recommendations
COMMENT 'Top 200 in-stock store-SKU pairs by predicted stockout risk with an LLM-written replenishment rationale for store managers (ai_query, system.ai.meta-llama-3-3-70b-instruct).'
AS
WITH top AS (
  SELECT * FROM serverless_stable_zlgeaw_catalog.stockout_ml.stockout_predictions
  WHERE NOT is_stockout
  ORDER BY risk_score DESC, days_of_cover ASC
  LIMIT 200
),
prompted AS (
  SELECT *, concat(
    'You are a retail replenishment assistant writing for a busy store manager. In 2-3 short sentences (max 70 words), ',
    'explain why this item is at risk of running out and what to order. Be specific with numbers, plain language, no greeting, no bullet points.\n',
    'Store: ', store_name, ' (', region, ')\nItem: ', sku_name, ' [', category, ']\nSupplier: ', supplier_name,
    ' - average lead time ', round(coalesce(avg_actual_lead_time_days, lead_time_days_mean), 1), ' days\n',
    'Stockout probability in next ', horizon_days, ' days: ', round(risk_score * 100), '%\n',
    'On hand: ', on_hand_units, ' units; on order: ', on_order_units, ' units; average demand: ', round(avg_demand_28d, 1),
    ' units/day; days of cover: ', round(days_of_cover, 1), '\n',
    'Promotion active: ', CASE WHEN on_promo THEN concat('yes (lift ', round(promo_lift, 2), 'x)') ELSE 'no' END, '\n',
    'Key risk drivers: ', array_join(key_drivers, '; '), '\n',
    CASE WHEN recommended_order_units = 0
         THEN concat('Recommended new order: 0 units, because ',
                     CASE WHEN capped_by_shelf THEN concat('on-hand plus on-order already fills the shelf capacity of ', shelf_capacity, ' units')
                          ELSE 'on-hand plus on-order already covers the target stock' END,
                     '. Do not tell the manager to order more; advise expediting the open order and checking the backroom.')
         ELSE concat('Recommended new order: ', recommended_order_units, ' units (case pack ', case_pack, ', shelf capacity ', shelf_capacity,
                     CASE WHEN capped_by_shelf THEN ', capped by shelf space' ELSE '' END, '). Use exactly this quantity.') END,
    '\nOnly use the numbers given above.') AS prompt
  FROM top
)
SELECT snapshot_date, store_id, store_name, region, sku_id, sku_name, category, supplier_id, supplier_name,
       risk_score, risk_tier, days_of_cover, on_hand_units, on_order_units, recommended_order_units, key_drivers,
       ai_query('system.ai.meta-llama-3-3-70b-instruct', prompt,
                modelParameters => named_struct('max_tokens', 160, 'temperature', 0.2)) AS rationale,
       'system.ai.meta-llama-3-3-70b-instruct' AS llm_endpoint,
       model_version, current_timestamp() AS generated_at
FROM prompted;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS rows, count(DISTINCT store_id, sku_id) AS unique_pairs,
       sum(CASE WHEN rationale IS NULL OR length(trim(rationale)) < 20 THEN 1 ELSE 0 END) AS empty_rationales,
       round(avg(length(rationale))) AS avg_chars, round(min(risk_score), 3) AS min_risk, round(max(risk_score), 3) AS max_risk
FROM serverless_stable_zlgeaw_catalog.stockout_ml.replenishment_recommendations;

rows,unique_pairs,empty_rationales,avg_chars,min_risk,max_risk
200,200,0,253.0,0.703,0.965


In [0]:
%sql
SELECT store_name, sku_name, risk_score, recommended_order_units, rationale
FROM serverless_stable_zlgeaw_catalog.stockout_ml.replenishment_recommendations
ORDER BY risk_score DESC LIMIT 5;

store_name,sku_name,risk_score,recommended_order_units,rationale
FreshMart #019,Household item 149,0.9647,24,"Household item 149 is at risk of running out with only 1.5 days of cover and a 96% stockout probability. We have 4 units on hand and 12 on order, but with average demand of 2.7 units/day, we need to order more. Order 24 units to prevent stockout."
FreshMart #038,Pantry item 135,0.9578,0,"Pantry item 135 is at risk of running out with only 1 unit on hand and 0.3 days of cover. Average demand is 3.1 units per day, exceeding the current stock. Order 12 units to replenish stock, but due to shelf capacity, no additional order is recommended beyond the 12 units already on order."
FreshMart #017,Household item 086,0.9566,24,"Household item 086 is at risk of running out with a 96% stockout probability in 7 days. We have only 0.6 days of cover and 5 units on hand, with average demand at 8.5 units/day. Order 24 units to mitigate the risk."
FreshMart #024,Household item 023,0.9566,48,"Household item 023 is at risk of running out with a 96% stockout probability in 7 days due to low on-hand stock of 1 unit and insufficient cover. Average demand is 5.3 units/day, exceeding current stock. Order 48 units to mitigate this risk."
FreshMart #024,Household item 077,0.9532,36,"Household item 077 is at risk of running out with only 0.5 days of cover and 2 units on hand. Average demand is 3.9 units/day, exceeding our current stock. Order 36 units to mitigate the 95% stockout probability within the next 7 days."
